# Media Consumption Analysis — Datenpipeline

**Von 1,45 Mio. Rohdaten-Headlines zu den Tableau-Aggregaten**

Dieses Notebook verarbeitet die beiden Rohdatensätze für das Projekt
`07_media_consumption_analysis`. Es liest die Daten ein, klassifiziert
Kategorien, harmonisiert Labels und erzeugt die vor-aggregierten
CSV-Dateien, die im Ordner `data/` liegen und die Grundlage für das
Tableau-Dashboard bilden.

**Quellen:**
- ABC News (AU): `Rohdaten/abcnews-date-text.csv` — nur Schlagzeilen, keine Kategorien
- HuffPost (US): `Rohdaten/News_Category_Dataset_v3.json` — Schlagzeilen + Kurzbeschreibung + native Kategorien + Autoren


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from collections import Counter
from pathlib import Path

pd.set_option('display.max_columns', None)

# Pfade
RAW_DIR = Path("Rohdaten")
OUT_DIR = Path("data")
OUT_DIR.mkdir(exist_ok=True)

ABC_PATH = RAW_DIR / "abcnews-date-text.csv"
HUFFPOST_PATH = RAW_DIR / "News_Category_Dataset_v3.json"

assert ABC_PATH.exists(), f"Nicht gefunden: {ABC_PATH}"
assert HUFFPOST_PATH.exists(), f"Nicht gefunden: {HUFFPOST_PATH}"


## 2. Rohdaten laden

In [ ]:
# --- ABC News (AU) ---
# Spalten im Original-Kaggle-Dataset: publish_date (YYYYMMDD), headline_text
abc_raw = pd.read_csv(ABC_PATH, dtype={"publish_date": str})
abc_raw["date"] = pd.to_datetime(abc_raw["publish_date"], format="%Y%m%d")
abc_raw = abc_raw.rename(columns={"headline_text": "headline"})
abc_raw["source"] = "ABC News (AU)"

print(f"ABC News: {len(abc_raw):,} Zeilen")
print(f"Zeitraum: {abc_raw['date'].min().date()} bis {abc_raw['date'].max().date()}")
abc_raw.head(3)


In [ ]:
# --- HuffPost (US) ---
# JSON Lines: category, headline, authors, link, short_description, date
hp_raw = pd.read_json(HUFFPOST_PATH, lines=True)
hp_raw["date"] = pd.to_datetime(hp_raw["date"])
hp_raw["source"] = "HuffPost (US)"

print(f"HuffPost: {len(hp_raw):,} Zeilen")
print(f"Zeitraum: {hp_raw['date'].min().date()} bis {hp_raw['date'].max().date()}")
hp_raw.head(3)


In [ ]:
combined_total = len(abc_raw) + len(hp_raw)
print(f"Kombiniert: {combined_total:,} Artikel")
print("Kanonischer Wert laut METHODOLOGY.md: 1.453.690")


## 3. Kategorisierung

**ABC News** liefert keine Kategorien — wir ordnen jede Headline per
Keyword-Zuordnung einer von 8 inhaltlichen Kategorien zu. Was nicht
eindeutig zugeordnet werden kann, fällt in `Sonstiges` (dokumentiertes
methodisches Limit, siehe `METHODOLOGY.md` Punkt 2a).

**HuffPost** hat bereits native Kategorien (z. B. `POLITICS`, `WELLNESS`,
`STYLE & BEAUTY`) — diese werden auf ein gemeinsames deutsches Vokabular
harmonisiert, damit beide Quellen vergleichbar sind.


In [ ]:
# --- 3a. Keyword-Regeln für ABC News ---
# Reihenfolge = Priorität (erste Übereinstimmung gewinnt), um Überschneidungen
# wie "drug" (Kriminalität vs. Gesundheit) kontrolliert aufzulösen.

ABC_KEYWORDS = {
    "Kriminalität": [
        r"\bmurder\w*\b", r"\bpolice\b", r"\bcourt\b", r"\bjail\b", r"\bprison\b",
        r"\barrest\w*\b", r"\bshoot\w*\b", r"\bstab\w*\b", r"\bassault\w*\b",
        r"\brobbery\b", r"\btheft\b", r"\bfraud\b", r"\bguilty\b", r"\bconvict\w*\b",
        r"\bhomicide\b", r"\btrial\b", r"\bcrime\b", r"\bcriminal\b", r"\bdrug charge",
    ],
    "Politik": [
        r"\bgovernment\b", r"\belection\w*\b", r"\bminister\w*\b", r"\bparliament\b",
        r"\bpresident\w*\b", r"\bsenate\b", r"\bvote\w*\b", r"\bpolicy\b",
        r"\bpolitician\b", r"\bcampaign\b", r"\breferendum\b", r"\blegislation\b",
        r"\bcabinet\b", r"\bpm\b", r"\bmp\b", r"\blabor party\b", r"\bliberal party\b",
    ],
    "Umwelt": [
        r"\bclimate\b", r"\benvironment\w*\b", r"\bweather\b", r"\bdrought\b",
        r"\bflood\w*\b", r"\bbushfire\w*\b", r"\bwildlife\b", r"\bpollution\b",
        r"\bemission\w*\b", r"\brenewable\b", r"\bscien\w*\b", r"\bresearch\b",
        r"\bnasa\b", r"\bspace\b", r"\bstudy\b",
    ],
    "Wirtschaft": [
        r"\beconomy\b", r"\beconomic\b", r"\bmarket\w*\b", r"\bbusiness\b",
        r"\btrade\b", r"\bbank\w*\b", r"\bshares?\b", r"\bstocks?\b", r"\bjobs?\b",
        r"\bunemploy\w*\b", r"\binflation\b", r"\bbudget\b", r"\btax\w*\b",
        r"\bcompany\b", r"\bindustry\b",
    ],
    "Sport": [
        r"\bcricket\b", r"\bfootball\b", r"\brugby\b", r"\btennis\b", r"\bolympic\w*\b",
        r"\bafl\b", r"\bnrl\b", r"\bmatch\b", r"\bcoach\b", r"\bchampionship\b",
        r"\btournament\b", r"\bgrand final\b", r"\bworld cup\b",
    ],
    "Welt": [
        r"\bworld\b", r"\binternational\b", r"\bforeign\b", r"\bwar\b", r"\bconflict\b",
        r"\bnation\w*\b", r"\bglobal\b", r"\bembassy\b", r"\btreaty\b", r"\bun\b",
        r"\bnato\b", r"\biraq\b", r"\bafghanistan\b", r"\bisrael\b", r"\bgaza\b",
    ],
    "Gesundheit": [
        r"\bhealth\w*\b", r"\bhospital\w*\b", r"\bdisease\b", r"\bvirus\b", r"\bcovid\w*\b",
        r"\bcancer\b", r"\bvaccin\w*\b", r"\bdoctor\w*\b", r"\bmedical\b", r"\bmental health\b",
        r"\bpatient\w*\b", r"\btreatment\b",
    ],
    "Gesellschaft": [
        r"\bcommunity\b", r"\bindigenous\b", r"\bsocial\b", r"\bculture\b", r"\breligio\w*\b",
        r"\beducation\b", r"\bfamily\b", r"\brights\b", r"\bprotest\w*\b", r"\brefugee\w*\b",
        r"\bimmigrat\w*\b",
    ],
}

# Vorkompilierte, kombinierte Regex pro Kategorie (schneller bei 1,2 Mio. Zeilen)
ABC_PATTERNS = {
    cat: re.compile("|".join(words), flags=re.IGNORECASE)
    for cat, words in ABC_KEYWORDS.items()
}

def classify_abc(headline: str) -> str:
    for cat, pattern in ABC_PATTERNS.items():
        if pattern.search(headline):
            return cat
    return "Sonstiges"

abc_raw["category"] = abc_raw["headline"].astype(str).apply(classify_abc)
print(abc_raw["category"].value_counts())


In [ ]:
# --- 3b. Kategorie-Harmonisierung für HuffPost ---
# Native HuffPost-Labels (Kaggle v3) -> kanonisches deutsches Vokabular

HUFFPOST_MAPPING = {
    "POLITICS": "Politik",
    "WELLNESS": "Gesundheit",
    "HEALTHY LIVING": "Gesundheit",
    "ENTERTAINMENT": "Entertainment",
    "COMEDY": "Entertainment",
    "TRAVEL": "Reisen",
    "STYLE & BEAUTY": "Style & Beauty",
    "STYLE": "Style & Beauty",
    "PARENTING": "Familie",
    "PARENTS": "Familie",
    "WEDDINGS": "Familie",
    "DIVORCE": "Familie",
    "FOOD & DRINK": "Essen",
    "TASTE": "Essen",
    "BUSINESS": "Wirtschaft",
    "MONEY": "Wirtschaft",
    "SPORTS": "Sport",
    "QUEER VOICES": "Gesellschaft",
    "BLACK VOICES": "Gesellschaft",
    "LATINO VOICES": "Gesellschaft",
    "WOMEN": "Gesellschaft",
    "IMPACT": "Gesellschaft",
    "MEDIA": "Gesellschaft",
    "HOME & LIVING": "Wohnen",
    "THE WORLDPOST": "Welt",
    "WORLDPOST": "Welt",
    "WORLD NEWS": "Welt",
    "U.S. NEWS": "US News",
    "CRIME": "Kriminalität",
    "GREEN": "Umwelt",
    "ENVIRONMENT": "Umwelt",
    "SCIENCE": "Umwelt",
    "RELIGION": "Religion",
    "ARTS": "Kunst",
    "ARTS & CULTURE": "Kunst",
    "CULTURE & ARTS": "Kunst",
    "COLLEGE": "Bildung",
    "EDUCATION": "Bildung",
    "TECH": "Sonstiges",
    "WEIRD NEWS": "Sonstiges",
    "FIFTY": "Sonstiges",
    "GOOD NEWS": "Sonstiges",
}

hp_raw["category"] = hp_raw["category"].map(HUFFPOST_MAPPING).fillna("Sonstiges")
print(hp_raw["category"].value_counts())


In [ ]:
# 00_kategorie_mapping.csv: dokumentiert die tatsächlich angewendete Harmonisierung
mapping_export = (
    pd.Series(HUFFPOST_MAPPING, name="kanonisches_label")
    .rename_axis("original_label")
    .reset_index()
    .sort_values("original_label")
)
mapping_export.to_csv(OUT_DIR / "00_kategorie_mapping.csv", index=False, encoding="utf-8-sig")
mapping_export.head()


## 4. Wortlängen & gemeinsames Long-Format

In [ ]:
def word_count(text):
    if pd.isna(text) or text == "":
        return np.nan
    return len(str(text).split())

abc_raw["headline_words"] = abc_raw["headline"].apply(word_count)
hp_raw["headline_words"] = hp_raw["headline"].apply(word_count)
hp_raw["description_words"] = hp_raw["short_description"].apply(word_count)
abc_raw["description_words"] = np.nan  # ABC hat keine Beschreibungstexte

abc_raw["year"] = abc_raw["date"].dt.year
abc_raw["month"] = abc_raw["date"].dt.month
hp_raw["year"] = hp_raw["date"].dt.year
hp_raw["month"] = hp_raw["date"].dt.month

common_cols = ["source", "date", "year", "month", "headline", "category",
               "headline_words", "description_words"]

combined = pd.concat([abc_raw[common_cols], hp_raw[common_cols]], ignore_index=True)
combined["monthname"] = combined["month"].apply(lambda m: f"{m:02d}-{pd.Timestamp(2000, m, 1).strftime('%b')}")

print(f"Kombiniert gesamt: {len(combined):,} Artikel")
combined.head(3)


## 5. `01_tableau_aggregiert.csv` — Verteilung, Trend, Saisonalität, Länge

In [ ]:
agg = (
    combined.groupby(["source", "year", "month", "monthname", "category"])
    .agg(
        artikel=("headline", "count"),
        med_headline_woerter=("headline_words", "median"),
        avg_headline_woerter=("headline_words", "mean"),
        med_beschreibung_woerter=("description_words", "median"),
    )
    .reset_index()
)
agg["avg_headline_woerter"] = agg["avg_headline_woerter"].round(2)

agg.to_csv(OUT_DIR / "01_tableau_aggregiert.csv", index=False, encoding="utf-8-sig")
print(f"{len(agg):,} Zeilen geschrieben")
agg.head(3)


## 6. `kategorie_summary.csv` — ABC-Kategorieverteilung

In [ ]:
abc_summary = (
    abc_raw["category"].value_counts()
    .rename_axis("category")
    .reset_index(name="anzahl")
)
abc_summary["anteil_pct"] = (abc_summary["anzahl"] / abc_summary["anzahl"].sum() * 100).round(1)

med_words = abc_raw.groupby("category")["headline_words"].median().rename("med_woerter")
abc_summary = abc_summary.merge(med_words, on="category")
abc_summary = abc_summary.sort_values("anzahl", ascending=False)

abc_summary.to_csv(OUT_DIR / "kategorie_summary.csv", index=False, encoding="utf-8-sig")
print(f"'Sonstiges'-Anteil: {abc_summary.loc[abc_summary['category']=='Sonstiges', 'anteil_pct'].values[0]}% "
      "(kanonisch laut METHODOLOGY.md: 60.0%)")
abc_summary


## 7. `02_tableau_keywords.csv` — Top-Keywords je Kategorie & Quelle

Einfache, robuste Tokenisierung: Kleinschreibung, nur Wörter (keine Zahlen/
Sonderzeichen), Stoppwörter entfernt. Pro Quelle & Kategorie werden die
Top-15 häufigsten Wörter ermittelt.


In [ ]:
STOPWORDS = set('''
a an the of to in for on with at by is are was were be been being this that
these those it its as from into over after before out up down about than then
and or but not no so if just says say said new us au also has have had will
'''.split())

TOKEN_RE = re.compile(r"[a-zA-Z]{3,}")

def top_keywords(texts, n=15):
    counter = Counter()
    for t in texts:
        if pd.isna(t):
            continue
        words = [w.lower() for w in TOKEN_RE.findall(str(t)) if w.lower() not in STOPWORDS]
        counter.update(words)
    return counter.most_common(n)

keyword_rows = []
for (src, cat), grp in combined.groupby(["source", "category"]):
    for rank, (kw, freq) in enumerate(top_keywords(grp["headline"]), start=1):
        keyword_rows.append({"source": src, "category": cat, "keyword": kw,
                              "haeufigkeit": freq, "rang": rank})

keywords_df = pd.DataFrame(keyword_rows)
keywords_df.to_csv(OUT_DIR / "02_tableau_keywords.csv", index=False, encoding="utf-8-sig")
print(f"{len(keywords_df):,} Zeilen geschrieben")
keywords_df.head(10)


## 8. `03_tableau_autoren.csv` — Autoren × Kategorie (HuffPost)

In [ ]:
hp_authors = hp_raw[["authors", "category"]].copy()
hp_authors["authors"] = hp_authors["authors"].fillna("").str.split(",")
hp_authors = hp_authors.explode("authors")
hp_authors["authors"] = hp_authors["authors"].str.strip()
hp_authors = hp_authors[hp_authors["authors"] != ""]

autoren = (
    hp_authors.groupby(["authors", "category"])
    .size()
    .reset_index(name="artikel")
    .rename(columns={"authors": "autor"})
    .sort_values("artikel", ascending=False)
)

autoren.to_csv(OUT_DIR / "03_tableau_autoren.csv", index=False, encoding="utf-8-sig")
print(f"{len(autoren):,} Zeilen, {autoren['autor'].nunique():,} Autor:innen")
autoren.head(5)


## 9. `04_tableau_vergleich.csv` & `VS_quellenvergleich.csv` — Quellenvergleich

Nur der **gemeinsame Zeitraum** und die **gemeinsamen Kategorien** beider
Quellen, je Quelle auf 100 % normiert (siehe `METHODOLOGY.md`, Punkt 2b).


In [ ]:
overlap_start = max(abc_raw["date"].min(), hp_raw["date"].min())
overlap_end = min(abc_raw["date"].max(), hp_raw["date"].max())
print(f"Gemeinsamer Zeitraum: {overlap_start.date()} bis {overlap_end.date()}")

overlap = combined[(combined["date"] >= overlap_start) & (combined["date"] <= overlap_end)]

shared_categories = (
    set(overlap.loc[overlap["source"] == "ABC News (AU)", "category"])
    & set(overlap.loc[overlap["source"] == "HuffPost (US)", "category"])
)
print(f"Gemeinsame Kategorien ({len(shared_categories)}): {sorted(shared_categories)}")

overlap_shared = overlap[overlap["category"].isin(shared_categories)]

counts = overlap_shared.groupby(["source", "category"]).size().reset_index(name="anzahl")
totals = counts.groupby("source")["anzahl"].transform("sum")
counts["anteil_prozent"] = (counts["anzahl"] / totals * 100).round(1)

vergleich = counts[["category", "source", "anteil_prozent"]].sort_values(["category", "source"])
vergleich.to_csv(OUT_DIR / "04_tableau_vergleich.csv", index=False, encoding="utf-8-sig")

# Breite Version fürs Dashboard (eine Zeile pro Kategorie, Quellen als Spalten)
vs_wide = vergleich.pivot(index="category", columns="source", values="anteil_prozent").reset_index()
vs_wide = vs_wide.rename(columns={"category": "scat"})
vs_wide.to_csv(OUT_DIR / "VS_quellenvergleich.csv", index=False, encoding="utf-8-sig")

vs_wide.sort_values("ABC News (AU)", ascending=False)


## 10. Validierung gegen `METHODOLOGY.md`

Diese Zelle prüft die berechneten Eckwerte gegen die in `METHODOLOGY.md`
dokumentierten Kennzahlen.


In [ ]:
canonical = {
    "Kombiniert gesamt": 1_453_690,
    "ABC News (AU)": 1_244_182,
    "HuffPost (US)": 209_508,
    "Sonstiges-Anteil ABC (%)": 60.0,
}

actual = {
    "Kombiniert gesamt": len(combined),
    "ABC News (AU)": len(abc_raw),
    "HuffPost (US)": len(hp_raw),
    "Sonstiges-Anteil ABC (%)": round(
        (abc_raw["category"] == "Sonstiges").mean() * 100, 1
    ),
}

comparison = pd.DataFrame({"dokumentiert": canonical, "berechnet": actual})
comparison["differenz"] = comparison["berechnet"] - comparison["dokumentiert"]
comparison


## 11. Fazit

Alle sieben Output-Dateien wurden in `data/` geschrieben:

- `00_kategorie_mapping.csv`
- `01_tableau_aggregiert.csv`
- `02_tableau_keywords.csv`
- `03_tableau_autoren.csv`
- `04_tableau_vergleich.csv`
- `VS_quellenvergleich.csv`
- `kategorie_summary.csv`

Diese Dateien bilden die Grundlage für das Tableau-Dashboard und die
Präsentation.
